Point of this script is to take Daft data and normalise it in a Listing and Seller class.

In [1]:
import pandas as pd
import matplotlib as plt
import json
import os
import glob
from parse import parse
from datetime import datetime, timezone

In [2]:
files = glob.glob(os.path.join("data", "*.jsonl"))
data = []
for file in files:
    with open(f'{file}', "r") as f:
        for line in f:
            d = json.loads(line)
            data.append(d)

In [3]:
print(len(data))

49


In [4]:
pages = [
    d.get('props',{}).get('pageProps', {}).get('listings', [])   
    for d in data
]

In [5]:
entries = []
for listings in pages:
    for entry in listings:
        listing = entry.get('listing')
        entries.append(listing)
    

In [41]:
# Prices work in a funny way in Daft
# Daft always takes lowest price and rounds up.
# ---> From €800 to €900 per month -> price is 800.0.
# ----> From €305 per week -> price is 310.0
# We need to normalise price, extract min, max, and cadence from price string. Then multiply all weekly values by 4 to normalise our price field.

from dataclasses import dataclass, field

@dataclass
class Listing:
    id: str
    publishedAt: datetime
    price_string: str
    price: float
    coordinates: list[float]
    bedrooms: str
    sellerId: str
    category: str
    max_price: float = field(init=False, default=None)
    min_price: float = field(init=False, default=None)
    monthly_price: float = field(init=False, default=None)
    cadence: str = field(init=False, default=None)
    

    def extract_pricing_with_parse(self, text):
        template_range = "From €{:,d} to €{:,d} per {}"
        template_single = "From €{:,d} per {}"
        simple_template = "€{:,d} per {}"

        match = parse(template_range, text)
        if match:
            return {"cadence": match[2], "min_price": match[0], "max_price": match[1]}   
            
        match = parse(template_single, text)
        if match:
            return {"cadence": match[1], "min_price": match[0], "max_price": match[0]}

        match = parse(simple_template, text)
        if match:
            return {"cadence": match[1], "min_price": match[0], "max_price": match[0]} 
        return {"cadence": None, "min_price": None, "max_price": None}

    def convert_price_string_float(self, text):
        if isinstance(text, str):
            return float(self.price.replace('€','').replace(',','').strip("+").strip(""))
        return text

    def __post_init__(self):
        price_dict = self.extract_pricing_with_parse(self.price_string)
        for key, value in price_dict.items():
            if key == "cadence":
                continue
            price_dict[key] = self.convert_price_string_float(value)
            
        self.max_price = price_dict.get('max_price', None)
        self.min_price = price_dict.get('min_price', None)
        self.cadence = price_dict.get('cadence', None)
        
        self.price = self.convert_price_string_float(self.price)
        monthly_price = None
        if self.cadence == "week":
            monthly_price = self.price * 4
            self.monthly_price = self.price * 4
        else:
            monthly_price = self.price

        self.monthly_price = monthly_price
        
        dt_utc = datetime.fromtimestamp(self.publishedAt / 1000, tz=timezone.utc)
        self.publishedAt = dt_utc

    def to_dict(self) -> dict[str]:
        return {
            "id": self.id,
            "publishedAt": self.publishedAt.strftime('%d/%m/%Y'),
            "price": self.price,
            "coordinates": self.coordinates,
            "price_string": self.price_string,
            "max_price": self.max_price,
            "min_price": self.min_price,
            "monthly_price": self.monthly_price,
            "bedrooms": self.bedrooms,
            "sellerId": self.sellerId,
            "category": self.category,
        }
            


        

In [14]:
# Map from Daft.ie to class
MAPPINGS = {
    "id": "id",
    "publishDate": "publishedAt",
    "price": "price_string",
    "abbreviatedPrice": "price",
    "point": "coordinates",
    "numBedrooms": "bedrooms",
    "seller": "sellerId",
    "category": "category"
}
def map_daft(entry):
    data = {}
    for key, value in MAPPINGS.items():
        if key in ['seller']:
            sellerId = entry.get(key, {}).get(value, None)
            data[value] = sellerId
        elif key in ["point"]:
            coordinates = entry.get(key, {}).get(value, [])
            data[value] = coordinates
        else:
            data[value] = entry.get(key)
    
    return data



In [8]:
entry_dicts = [map_daft(entry) for entry in entries]
entry_dicts[0]

{'id': 6618514,
 'publishedAt': 1789732220000,
 'price_string': 'From €305 per week',
 'price': '€310+',
 'coordinates': [-6.275637647916881, 53.33777790919868],
 'bedrooms': '1 bed',
 'sellerId': 13963,
 'category': 'Rent'}

In [44]:
entries = [Listing(**entry) for entry in entry_dicts]

In [71]:
y = []
for i in entries:
    x = i.to_dict()
    y.append(x.get('bedrooms'))

In [79]:
s = pd.Series(y)
s.unique()

array(['1 bed', 'Single Room', 'Double Room', 'Single & Double Room',
       'Shared Room', 'Single & Twin Room', 'Twin Room',
       'Single, Double & Shared Room', 'Twin & Shared Room',
       'Double & Twin Room', 'Single, Twin & Shared Room'], dtype=object)